# Forge — LoRA Training (SFT / Instruction)

**Session:** `3fdc8d19-ff80-4ede-a63a-b36c0f57b797`  
**Base model:** `mistralai/Mistral-7B-Instruct-v0.3`  
**Dataset:** 10 examples from `samokmin/Python@main`  
**Dataset path:** `sessions/3fdc8d19-ff80-4ede-a63a-b36c0f57b797/data.jsonl`  
**Recipe:** `sessions/3fdc8d19-ff80-4ede-a63a-b36c0f57b797/config.json`

## Before you run
1. **Runtime → Change runtime type → T4 GPU** (or better).
2. If your repo is **private**, replace the raw URL below with a `git clone https://<TOKEN>@github.com/...` cell.
3. For **gated** base models (Llama, Gemma, Mistral, …):
   - Go to the model page on Hugging Face (e.g. https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3) and **click "Agree and access repository"**. Wait until status shows "You have been granted access".
   - Create a token at https://huggingface.co/settings/tokens (type: **Read**, and tick **"Access public gated repos"**).
   - In Colab, click the 🔑 **Secrets** icon in the left sidebar → add a secret named `HF_TOKEN` → paste your token → enable **Notebook access**.

Everything else is configured — just **Runtime → Run all**.

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl huggingface_hub

## 2. Hugging Face login (for gated models)
Reads `HF_TOKEN` from Colab Secrets. If you don't use a gated model, this cell does nothing.

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')

if token:
    from huggingface_hub import login
    login(token=token, add_to_git_credential=True)
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('✅ Logged in to Hugging Face.')
else:
    print('⚠️ No HF_TOKEN found. If your base model is gated (Llama/Gemma/Mistral), add it via Colab Secrets (🔑) and re-run this cell.')

## 3. Load recipe + dataset from GitHub

In [ ]:
import json, urllib.request
from datasets import load_dataset

CONFIG_URL  = "https://raw.githubusercontent.com/samokmin/Python/main/sessions/3fdc8d19-ff80-4ede-a63a-b36c0f57b797/config.json"
DATASET_URL = "https://raw.githubusercontent.com/samokmin/Python/main/sessions/3fdc8d19-ff80-4ede-a63a-b36c0f57b797/data.jsonl"

with urllib.request.urlopen(CONFIG_URL) as r:
    CFG = json.loads(r.read().decode('utf-8'))

print('Recipe:', json.dumps(CFG, indent=2, ensure_ascii=False))

raw_ds = load_dataset('json', data_files=DATASET_URL, split='train')
print(raw_ds)

## 4. Format examples as ChatML
Each row becomes a `messages` conversation: system → user (prompt) → assistant (response).

In [ ]:
DEFAULT_SYSTEM = 'You are a helpful assistant.'

def to_messages(row):
    return {
        'messages': [
            {'role': 'system', 'content': DEFAULT_SYSTEM},
            {'role': 'user', 'content': row.get('prompt', '')},
            {'role': 'assistant', 'content': row.get('completion', row.get('response', ''))},
        ]
    }

train_ds = raw_ds.map(to_messages, remove_columns=raw_ds.column_names)
print(train_ds[0])

## 5. Load base model in 4-bit + tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)

## 6. Attach LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

adapter_cfg = CFG.get('adapter', {})
lora_cfg = LoraConfig(
    r=int(adapter_cfg.get('r', 16)),
    lora_alpha=int(adapter_cfg.get('alpha', 32)),
    lora_dropout=float(adapter_cfg.get('dropout', 0.05)),
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=adapter_cfg.get('target_modules', ['q_proj','k_proj','v_proj','o_proj']),
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## 7. Train (SFTTrainer)

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_cfg = SFTConfig(
    output_dir='./lora-adapter',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy='epoch',
    bf16=False,
    fp16=True,
    gradient_checkpointing=False,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    args=sft_cfg,
)

trainer.train()

## 8. Save & download the adapter

In [ ]:
trainer.save_model('./lora-adapter')
tokenizer.save_pretrained('./lora-adapter')

import shutil
shutil.make_archive('lora-adapter', 'zip', './lora-adapter')

try:
    from google.colab import files
    files.download('lora-adapter.zip')
except Exception as e:
    print('Skipping download (not in Colab):', e)

## 9. Smoke test — generate from the fresh adapter

In [ ]:
from transformers import pipeline

test_prompt = raw_ds[0].get('prompt', 'Hello!')
gen = pipeline('text-generation', model=model, tokenizer=tokenizer)
out = gen(test_prompt, max_new_tokens=200, do_sample=True, temperature=0.7)
print(out[0]['generated_text'])

## 10. Interactive testing — chat with your adapter
Run this cell to enter an interactive loop. Type a prompt, press Enter to generate.
Type `quit` (or just press Enter with empty input) to exit.

Works both with the freshly trained adapter (in memory) **and** with an adapter already pushed to the Hub.
To load an adapter from the Hub instead, set `ADAPTER_REPO` to e.g. `'min-samis2/Jonas_Biliunas'` and re-run this cell.

In [ ]:
# Set to a HF repo id to load an adapter from the Hub instead of using the in-memory one.
ADAPTER_REPO = None  # e.g. 'min-samis2/Jonas_Biliunas'

if ADAPTER_REPO:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb,
        device_map='auto',
        trust_remote_code=True,
    )
    infer_model = PeftModel.from_pretrained(base, ADAPTER_REPO)
    print(f'✅ Loaded adapter {ADAPTER_REPO} on top of {BASE_MODEL}')
else:
    infer_model = model
    print('✅ Using in-memory adapter from this session')

chat = pipeline('text-generation', model=infer_model, tokenizer=tokenizer)

print('\n=== Interactive mode — type "quit" to exit ===\n')
while True:
    try:
        prompt = input('You ▸ ').strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not prompt or prompt.lower() in {'quit', 'exit'}:
        break
    out = chat(prompt, max_new_tokens=300, do_sample=True, temperature=0.7, top_p=0.9)
    text = out[0]['generated_text']
    if text.startswith(prompt):
        text = text[len(prompt):].lstrip()
    print(f'\nModel ▸ {text}\n')

## 11. Serve adapter as a public API (for the Lovable app)
Runs a small FastAPI server inside this Colab and opens an ngrok tunnel so the **My models** playground can call it.

**One-time setup:**
1. Get a free ngrok token: https://dashboard.ngrok.com/get-started/your-authtoken
2. In Colab → 🔑 **Secrets** → add `NGROK_TOKEN` → enable Notebook access.

When you run this cell, it prints a public URL like `https://abcd-34-83.ngrok-free.app`. Copy it into the **Colab endpoint URL** field on the My models page. Keep this cell running — the URL stops working as soon as you stop it.

In [ ]:
!pip install -q fastapi uvicorn nest_asyncio pyngrok

import os, nest_asyncio, uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok, conf
from transformers import pipeline

NGROK_TOKEN = None  # or set as Colab Secret 'NGROK_TOKEN'
try:
    from google.colab import userdata
    NGROK_TOKEN = NGROK_TOKEN or userdata.get('NGROK_TOKEN')
except Exception:
    NGROK_TOKEN = NGROK_TOKEN or os.environ.get('NGROK_TOKEN')
if not NGROK_TOKEN:
    raise RuntimeError('Set NGROK_TOKEN as a Colab Secret (🔑) and re-run this cell.')
conf.get_default().auth_token = NGROK_TOKEN

# Reuse the model loaded in cell 10 (or fall back to the freshly trained one)
serve_model = globals().get('infer_model', model)
gen = pipeline('text-generation', model=serve_model, tokenizer=tokenizer)

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

class GenIn(BaseModel):
    prompt: str
    system: str | None = ''
    max_new_tokens: int = 300
    temperature: float = 0.7

@app.get('/health')
def health():
    return {'ok': True, 'base_model': BASE_MODEL}

@app.post('/generate')
def generate(inp: GenIn):
    full = (inp.system + '\n\n' if inp.system else '') + inp.prompt
    out = gen(full, max_new_tokens=inp.max_new_tokens, do_sample=True,
              temperature=max(inp.temperature, 0.01), top_p=0.9)
    text = out[0]['generated_text']
    if text.startswith(full):
        text = text[len(full):].lstrip()
    return {'text': text}

for t in ngrok.get_tunnels():
    try: ngrok.disconnect(t.public_url)
    except Exception: pass
public_url = ngrok.connect(8000, 'http').public_url
print('\n' + '=' * 60)
print(f'🌐 Paste this URL into the Lovable app (My models → Colab endpoint URL):')
print(f'   {public_url}')
print('=' * 60 + '\n')

nest_asyncio.apply()
uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

## 12. (Optional) Push adapter to Hugging Face Hub
Requires `HF_TOKEN` with **write** permission (step 2).

In [ ]:
# model.push_to_hub('your-username/your-adapter-name')
# tokenizer.push_to_hub('your-username/your-adapter-name')

## 13. (Optional) Merge LoRA into base model and push as a full model
Use this if you want to test via the HF Inference Router instead of the Colab tunnel.
**Warning:** the merged model is ~16 GB and upload is slow. Most users should stick with step 11.

In [ ]:
# merged = model.merge_and_unload()
# merged.save_pretrained('./merged-model', safe_serialization=True)
# tokenizer.save_pretrained('./merged-model')
# merged.push_to_hub('your-username/your-model-merged')
# tokenizer.push_to_hub('your-username/your-model-merged')